In [1]:
!mkdir ~/.kaggle

In [2]:
!mv kaggle.json ~/.kaggle/

In [3]:
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download -d kaggleprollc/phishing-url-websites-dataset-phiusiil

Dataset URL: https://www.kaggle.com/datasets/kaggleprollc/phishing-url-websites-dataset-phiusiil
License(s): CC-BY-SA-4.0
 32% 5.00M/15.9M [00:00<00:00, 29.9MB/s]
100% 15.9M/15.9M [00:00<00:00, 72.7MB/s]


In [5]:
!unzip phishing-url-websites-dataset-phiusiil.zip

Archive:  phishing-url-websites-dataset-phiusiil.zip
  inflating: PhiUSIIL_Phishing_URL_Dataset.csv  


In [6]:
import pandas as pd

In [7]:
df = pd.read_csv('/content/PhiUSIIL_Phishing_URL_Dataset.csv')

In [8]:
df = df[['URL', 'label']]

In [9]:
df.head()

,URL,label
0,https://www.southbankmosaics.com,1
1,https://www.uni-mainz.de,1
2,https://www.voicefmradio.co.uk,1
3,https://www.sfnmjournal.com,1
4,https://www.rewildingargentina.org,1


In [10]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   URL     235795 non-null  object
 1   label   235795 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 3.6+ MB


In [12]:
df.duplicated().sum()

425

In [13]:
# Drop duplicate rows from the DataFrame
df = df.drop_duplicates()
df.shape

(235370, 2)

In [14]:
df['label'].value_counts()

,count
label,
1,134850
0,100520


In [16]:
from transformers import BertTokenizer

In [17]:
# Load the BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [18]:
# Tokenize the text column
inputs = tokenizer(df['URL'].tolist(), return_tensors='pt', padding=True, truncation=True)

In [19]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
import torch

In [21]:
# Load pre-trained BERT for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=len(df['label'].unique()))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [22]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',           # Output directory
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=200,
    fp16=True,                         # Enable mixed precision training for speed-up
    gradient_accumulation_steps=1,     # Accumulate gradients over 2 steps
    save_total_limit=1,                # Keep only the most recent checkpoint
    evaluation_strategy="no",
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [23]:
class APIDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [24]:
labels = df['label'].tolist()

In [25]:
dataset = APIDataset(inputs, labels)

In [26]:
# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
)

/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [27]:
# Train the model
trainer.train()

<ipython-input-23-990668a8708f>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


Step,Training Loss
200,0.138100
400,0.016900
600,0.025600
800,0.013500
1000,0.018700
1200,0.011500
1400,0.008200
1600,0.016100
1800,0.035200
2000,0.021000


<ipython-input-23-990668a8708f>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-23-990668a8708f>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-23-990668a8708f>:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-23-990668a8708f>:7: UserWarning: To copy construct from a 

TrainOutput(global_step=14711, training_loss=0.02051342079425667, metrics={'train_runtime': 6592.016, 'train_samples_per_second': 35.705, 'train_steps_per_second': 2.232, 'total_flos': 6.19284491000832e+16, 'train_loss': 0.02051342079425667, 'epoch': 1.0})

In [28]:
# Save the model
model.save_pretrained('./saved_model')

# Save the tokenizer
tokenizer.save_pretrained('./saved_model')

('./saved_model/tokenizer_config.json',
 './saved_model/special_tokens_map.json',
 './saved_model/vocab.txt',
 './saved_model/added_tokens.json')

In [29]:
input_text = "www.google.com"

In [32]:
input_text2 = "..........."

In [34]:
input_text3 = "http://google.com-redirect@valimail.com"

In [36]:
input_text4 = 'esxcc.com/js/index.htm?us.battle.net/noghn/en/?ref5M+9d'

In [30]:
# Tokenize and prepare the input
inputs = tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=512)

# Move the inputs to the correct device (CPU/GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
inputs = {key: val.to(device) for key, val in inputs.items()}

# Put the model in evaluation mode and move it to the correct device
model.eval()
model.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get the predicted label (the index of the maximum logit)
predicted_label = torch.argmax(logits, dim=-1).item()

# Output the predicted label
print(f'Predicted label: {predicted_label}')

Predicted label: 1


In [33]:
# Tokenize and prepare the input
inputs = tokenizer(input_text2, return_tensors="pt", padding=True, truncation=True, max_length=512)

# Move the inputs to the correct device (CPU/GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
inputs = {key: val.to(device) for key, val in inputs.items()}

# Put the model in evaluation mode and move it to the correct device
model.eval()
model.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get the predicted label (the index of the maximum logit)
predicted_label = torch.argmax(logits, dim=-1).item()

# Output the predicted label
print(f'Predicted label: {predicted_label}')

Predicted label: 1


In [35]:
# Tokenize and prepare the input
inputs = tokenizer(input_text3, return_tensors="pt", padding=True, truncation=True, max_length=512)

# Move the inputs to the correct device (CPU/GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
inputs = {key: val.to(device) for key, val in inputs.items()}

# Put the model in evaluation mode and move it to the correct device
model.eval()
model.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get the predicted label (the index of the maximum logit)
predicted_label = torch.argmax(logits, dim=-1).item()

# Output the predicted label
print(f'Predicted label: {predicted_label}')

Predicted label: 0


In [37]:
# Tokenize and prepare the input
inputs = tokenizer(input_text4, return_tensors="pt", padding=True, truncation=True, max_length=512)

# Move the inputs to the correct device (CPU/GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
inputs = {key: val.to(device) for key, val in inputs.items()}

# Put the model in evaluation mode and move it to the correct device
model.eval()
model.to(device)

# Make predictions
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Get the predicted label (the index of the maximum logit)
predicted_label = torch.argmax(logits, dim=-1).item()

# Output the predicted label
print(f'Predicted label: {predicted_label}')

Predicted label: 0
